In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
df=pd.read_csv('train.csv')
df.head()

In [ ]:
df['Age']=df['Age'].fillna(df['Age'].median())
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])

In [ ]:
y=df.iloc[:,1].values
X=df[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']].values
X

In [ ]:
y

In [ ]:
Embarked_C=[]
Embarked_Q=[]
Embarked_S=[]
for i in range(X.shape[0]):
    if (X[i][1]=='male'):
        X[i][1]=1
    else:
        X[i][1]=0
    if (X[i][-1]=='C'):
        Embarked_C.append(1)
        Embarked_Q.append(0)
        Embarked_S.append(0)
    elif (X[i][-1]=='Q'):
        Embarked_C.append(0)
        Embarked_Q.append(1)
        Embarked_S.append(0)
    elif (X[i][-1]=='S'):
        Embarked_C.append(0)
        Embarked_Q.append(0)
        Embarked_S.append(1)
Embarked_C=np.array(Embarked_C)
Embarked_Q=np.array(Embarked_Q)
Embarked_S=np.array(Embarked_S)
X=X[:,:-1]
X=np.column_stack((X,Embarked_C,Embarked_Q,Embarked_S))

In [ ]:
scaler=StandardScaler()
X[:,[0,2,3,4,5]]=scaler.fit_transform(X[:,[0,2,3,4,5]])

In [ ]:
X.shape

In [ ]:
X[:,8]

In [ ]:
X[0]

In [ ]:
X=X.astype(np.float32)
y=y.astype(np.float32)

In [ ]:
model=Sequential([
    Dense(units=19, activation='relu'),
    Dense(units=7, activation='relu'),
    Dense(units=1, activation='linear')
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),metrics=['accuracy'])
history=model.fit(X,y,epochs=100)

In [ ]:
print("Final loss: ",history.history['loss'][-1])
print("Final accuracy: ",history.history['accuracy'][-1])

In [ ]:
tests=pd.read_csv('train.csv')
tests['Age']=tests['Age'].fillna(tests['Age'].median())
tests['Embarked']=tests['Embarked'].fillna(tests['Embarked'].mode()[0])
test=tests[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']].values
Embarked_C=[]
Embarked_Q=[]
Embarked_S=[]
for i in range(test.shape[0]):
    if (test[i][1]=='male'):
        test[i][1]=1
    else:
        test[i][1]=0
    if (test[i][-1]=='C'):
        Embarked_C.append(1)
        Embarked_Q.append(0)
        Embarked_S.append(0)
    elif (test[i][-1]=='Q'):
        Embarked_C.append(0)
        Embarked_Q.append(1)
        Embarked_S.append(0)
    elif (test[i][-1]=='S'):
        Embarked_C.append(0)
        Embarked_Q.append(0)
        Embarked_S.append(1)
Embarked_C=np.array(Embarked_C)
Embarked_Q=np.array(Embarked_Q)
Embarked_S=np.array(Embarked_S)
test=test[:,:-1]
test=np.column_stack((test,Embarked_C,Embarked_Q,Embarked_S))
test=test.astype(np.float32)

In [ ]:
pred=model(test)
pred=tf.sigmoid(pred).numpy().flatten()
pred=(pred>0.5).astype(int)

In [ ]:
sub = pd.DataFrame({
    'PassengerId': tests['PassengerId'],
    'Survived': pred
})
sub.to_csv('submission.csv', index=False)
print(f'Saved: {sub.shape}, survival rate: {sub["Survived"].mean():.3f}')
print(sub.head(10))